# Operations Lakehouse · hallazgos de Gold

Consultas sobre la publicación vigente de Gold.

In [ ]:
workspace_name = "operations-lakehouse"
lakehouse_name = "lh_operations"

In [ ]:
from lakehouse.fabric import fabric_paths
from lakehouse.gold import current_publication, register_views

paths = fabric_paths(workspace_name, lakehouse_name)
print("Publicación vigente:", current_publication(spark, paths))
register_views(spark, paths)

## Excursiones críticas en cuartos fríos

In [ ]:
display(spark.sql("""
    SELECT room_code, start_utc, end_utc, duration_min, peak_temp_c, max_temp_c
    FROM gold_fact_cold_room_excursion
    WHERE severity = 'CRITICA'
    ORDER BY start_utc
"""))

## Estado de cadena de frío por lote

In [ ]:
display(spark.sql("""
    SELECT cold_chain_status, COUNT(*) AS lots, SUM(qty_kg) AS qty_kg
    FROM gold_fact_lot_cold_chain
    GROUP BY cold_chain_status
    ORDER BY cold_chain_status
"""))

## Lotes comprometidos

In [ ]:
display(spark.sql("""
    SELECT lot_code, product_name, qty_kg, cold_room_code, shipment_number,
           critical_exposure_min, transit_out_of_range_h
    FROM gold_fact_lot_cold_chain
    WHERE cold_chain_status = 'COMPROMETIDO'
    ORDER BY lot_code
"""))

## Contenedores con temperatura fuera de rango o retraso

In [ ]:
display(spark.sql("""
    SELECT shipment_number, container_number, port_of_discharge, status,
           arrival_delay_h, transit_out_of_range_h, max_supply_air_c
    FROM gold_fact_container_transit
    WHERE transit_out_of_range_h > 0 OR arrival_delay_h >= 24
    ORDER BY arrival_delay_h DESC
"""))